# Retail Sales Analysis with PySpark

A small learning project: processing ~1M rows of real online-retail transactions with PySpark.

**Dataset:** [Online Retail II (UCI)](https://archive.ics.uci.edu/dataset/502/online+retail+ii): transactions of a UK-based online retailer, Dec 2009 – Dec 2011.

**What this notebook covers:**
1. Creating a SparkSession and reading data with an explicit schema
2. Cleaning: nulls, cancellations, invalid quantities/prices, duplicates
3. Aggregations with `groupBy`
4. Weekly demand per product with **window functions** (moving average, lag, week-over-week change)
5. The same kind of analysis in **Spark SQL**
6. Writing results to **Parquet**

## 0. Setup

Requirements: Python 3.10+, Java 17 or 21, and `pip install pyspark pandas openpyxl`.

Download `online_retail_II.xlsx` from the UCI link above and put it in a `data/` folder next to this notebook.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = (
    SparkSession.builder
    .appName("RetailSalesPySpark")
    .master("local[*]")                          # use all local CPU cores
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "8")  # default is 200; too many for a small local dataset
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark.version

## 1. Convert Excel to CSV (one-time)

Spark cannot read `.xlsx` natively, so we convert it once with pandas. The file has two sheets (2009–2010 and 2010–2011); we combine them.
This step can take a few minutes. It is skipped if the CSV already exists.

In [ ]:
import os
import pandas as pd

XLSX_PATH = "data/online_retail_II.xlsx"
CSV_PATH = "data/online_retail_II.csv"

if not os.path.exists(CSV_PATH):
    sheets = pd.read_excel(
        XLSX_PATH, sheet_name=None,
        dtype={"Invoice": str, "StockCode": str, "Customer ID": str},
    )
    pd.concat(sheets.values(), ignore_index=True).to_csv(CSV_PATH, index=False)
    print("CSV written:", CSV_PATH)
else:
    print("CSV already exists, skipping conversion")

## 2. Read with an explicit schema

Defining the schema ourselves is faster than `inferSchema=True` (Spark doesn't need an extra pass over the data) and avoids wrong type guesses.
`InvoiceDate` is read as a string first and parsed afterwards, so the notebook works with both the pandas-converted CSV and common CSV versions found online.

In [ ]:
schema = StructType([
    StructField("Invoice", StringType()),
    StructField("StockCode", StringType()),
    StructField("Description", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("InvoiceDate", StringType()),
    StructField("Price", DoubleType()),
    StructField("Customer ID", StringType()),
    StructField("Country", StringType()),
])

raw = (
    spark.read.csv(CSV_PATH, header=True, schema=schema)
    .withColumnRenamed("Customer ID", "CustomerID")
    .withColumn(
        "InvoiceDate",
        F.coalesce(
            F.try_to_timestamp("InvoiceDate", F.lit("yyyy-MM-dd HH:mm:ss")),
            F.try_to_timestamp("InvoiceDate", F.lit("M/d/yyyy H:mm")),
        ),
    )
)

raw.printSchema()
raw.show(5, truncate=False)
print("Raw rows:", raw.count())

## 3. Cleaning

Rules:
- Drop rows without a customer ID
- Drop cancellations (invoice numbers starting with `C`)
- Keep only positive quantities and prices
- Drop exact duplicate rows
- Add a `Revenue` column

Spark is **lazy**: these transformations only build a plan; nothing runs until an action like `count()` or `show()`. We `cache()` the clean data because we reuse it many times below.

In [ ]:
clean = (
    raw
    .filter(F.col("CustomerID").isNotNull())
    .filter(F.col("InvoiceDate").isNotNull())
    .filter(~F.col("Invoice").startswith("C"))
    .filter((F.col("Quantity") > 0) & (F.col("Price") > 0))
    .dropDuplicates()
    .withColumn("Revenue", F.round(F.col("Quantity") * F.col("Price"), 2))
    .cache()
)

n_raw, n_clean = raw.count(), clean.count()
print(f"Raw: {n_raw:,} | Clean: {n_clean:,} | Removed: {n_raw - n_clean:,} ({(n_raw - n_clean) / n_raw:.1%})")

## 4. Aggregations with `groupBy`

In [ ]:
# Top 10 products by revenue
top_products = (
    clean.groupBy("StockCode")
    .agg(
        F.first("Description", ignorenulls=True).alias("Description"),
        F.sum("Quantity").alias("TotalQuantity"),
        F.round(F.sum("Revenue"), 2).alias("TotalRevenue"),
        F.countDistinct("Invoice").alias("NumOrders"),
    )
    .orderBy(F.desc("TotalRevenue"))
)
top_products.show(10, truncate=False)

In [ ]:
# Revenue by country, with each country's share of total revenue
total_revenue = clean.agg(F.sum("Revenue")).first()[0]

by_country = (
    clean.groupBy("Country")
    .agg(
        F.round(F.sum("Revenue"), 2).alias("Revenue"),
        F.countDistinct("CustomerID").alias("Customers"),
    )
    .withColumn("RevenueShare", F.round(F.col("Revenue") / total_revenue, 4))
    .orderBy(F.desc("Revenue"))
)
by_country.show(10, truncate=False)

## 5. Weekly demand per product with window functions

First we aggregate sales to **product × week**. Then, with a window partitioned by product and ordered by week, we compute:
- `MA4`: 4-week moving average of quantity (current week + previous 3)
- `PrevWeekQty`: last week's quantity (`lag`)
- `WoWChange`: week-over-week % change

**Limitation:** weeks in which a product sold nothing do not appear as rows, so `lag` and the moving average skip them. For real demand forecasting you would fill missing weeks with 0 first.

In [ ]:
weekly = (
    clean
    .withColumn("Week", F.date_trunc("week", "InvoiceDate").cast("date"))
    .groupBy("StockCode", "Week")
    .agg(
        F.sum("Quantity").alias("WeeklyQty"),
        F.round(F.sum("Revenue"), 2).alias("WeeklyRevenue"),
    )
)

w = Window.partitionBy("StockCode").orderBy("Week")

weekly_features = (
    weekly
    .withColumn("MA4", F.round(F.avg("WeeklyQty").over(w.rowsBetween(-3, 0)), 2))
    .withColumn("PrevWeekQty", F.lag("WeeklyQty", 1).over(w))
    .withColumn(
        "WoWChange",
        F.round((F.col("WeeklyQty") - F.col("PrevWeekQty")) / F.col("PrevWeekQty"), 3),
    )
)

# Look at the best-selling product
best = top_products.first()["StockCode"]
weekly_features.filter(F.col("StockCode") == best).orderBy("Week").show(12)

In [ ]:
# Company-wide weekly revenue trend
weekly_total = (
    weekly.groupBy("Week")
    .agg(F.round(F.sum("WeeklyRevenue"), 2).alias("Revenue"))
    .orderBy("Week")
)
weekly_total.show(10)

# Optional: plot it (small result, so converting to pandas is fine)
import matplotlib.pyplot as plt
pdf = weekly_total.toPandas()
plt.figure(figsize=(10, 4))
plt.plot(pdf["Week"], pdf["Revenue"])
plt.title("Weekly revenue")
plt.xlabel("Week")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

## 6. Spark SQL

The same DataFrame can be queried with SQL by registering it as a temporary view. Here: monthly revenue for the top 5 countries.

In [ ]:
clean.createOrReplaceTempView("sales")

monthly_top_countries = spark.sql("""
    WITH top_countries AS (
        SELECT Country
        FROM sales
        GROUP BY Country
        ORDER BY SUM(Revenue) DESC
        LIMIT 5
    )
    SELECT
        s.Country,
        DATE_TRUNC('month', s.InvoiceDate) AS Month,
        ROUND(SUM(s.Revenue), 2)           AS Revenue
    FROM sales s
    JOIN top_countries t ON s.Country = t.Country
    GROUP BY s.Country, DATE_TRUNC('month', s.InvoiceDate)
    ORDER BY s.Country, Month
""")
monthly_top_countries.show(10, truncate=False)

## 7. Save results as Parquet

Parquet is a columnar format: smaller than CSV, keeps the schema, and is fast to read back in Spark.

In [ ]:
weekly_features.write.mode("overwrite").parquet("output/weekly_product_features")
by_country.write.mode("overwrite").parquet("output/revenue_by_country")

# Read back to check
spark.read.parquet("output/weekly_product_features").show(5)

In [ ]:
spark.stop()